# Quaternion filtering

If $q: \mathbb{R} \to \mathbb{R}^4$ is a path through the unit quaternions, then we can write
$$
  \dot{q}(t) = q(t) \, \omega(t), \qquad \omega = 0 + \omega_1 \, i + \omega_2 \, j + \omega_3 \, k
$$
where $2 \, \omega$ is the usual angular velocity.
There is a question about how to filter such a curve $q$ using linear filters.
One approach is to filter the components of $q$ and then normalize the output.
With respect to optimization, this introduces another control variable, which isn't necessarily desirable.
Another solution is to filter the angular velocity $\omega$, which induces a path through the unit quaternions.
Moreover, $\omega$ gives better intuition about how to move through the space of rotations (it obviously encodes direction and distance).
These seem like desirable properties.
It might be useful to try implementing this.

In [ ]:
import functools
from IPython.display import display
import sympy as sp
import numpy as np
import scipy.integrate as sci_int
import control as ct
import jax
import jax.numpy as jnp

import exp_mpc.stewart_min.siso as siso

jax.config.update("jax_enable_x64", True)

In [ ]:
def quat2list(q):
    return [q.a, q.b, q.c, q.d]

q2l = quat2list

In [ ]:
q = sp.Quaternion(*[sp.Symbol(f"q_{i}", real=True) for i in range(4)])
v = sp.Quaternion(*[0.0] + [sp.Symbol(f"v_{i}", real=True) for i in range(1, 4)])

## The ZoH matrix

In [ ]:
eqs = quat2list(q * v)
A = sp.Matrix(4, 4, lambda i, j: eqs[i].coeff(quat2list(q)[j]))
A_exp = A.exp()

In [ ]:
v_norm_2 = v.b**2 + v.c**2 + v.d**2
sub_expr0_pos = sp.exp(sp.sqrt(-v_norm_2)) / sp.sqrt(v_norm_2)
sub_expr0_neg = sp.exp(-sp.sqrt(-v_norm_2)) / sp.sqrt(v_norm_2)

re_sub0_pos = sp.re(sub_expr0_pos)
im_sub0_pos = sp.im(sub_expr0_pos)
re_sub0_neg = sp.re(sub_expr0_neg)
im_sub0_neg = sp.im(sub_expr0_neg)

re_sub1_pos = sp.cos(sp.sqrt(v_norm_2)) / sp.sqrt(v_norm_2)
im_sub1_pos = sp.sin(sp.sqrt(v_norm_2)) / sp.sqrt(v_norm_2)
re_sub1_neg = sp.cos(-sp.sqrt(v_norm_2)) / sp.sqrt(v_norm_2)
im_sub1_neg = sp.sin(-sp.sqrt(v_norm_2)) / sp.sqrt(v_norm_2)

In [ ]:
A_exp_simp = sp.simplify(A_exp.xreplace({
    re_sub0_pos: re_sub1_pos,
    im_sub0_pos: im_sub1_pos,
    re_sub0_neg: re_sub1_neg,
    im_sub0_neg: im_sub1_neg,
}))
v_norm = sp.Symbol(r"\|v\|", real=True)
A_zoh = A_exp_simp.subs({sp.sqrt(v_norm_2): v_norm})
A_zoh

In [ ]:
omega = sp.Quaternion(sp.cos(v_norm), v.b * sp.sin(v_norm) / v_norm, v.c * sp.sin(v_norm) / v_norm, v.d * sp.sin(v_norm) / v_norm)
rot = q * omega
assert np.allclose(np.array(list(sp.Matrix(4, 4, lambda i, j: quat2list(rot)[i].coeff(quat2list(q)[j])) - A_zoh), dtype=float), 0.0)

In [ ]:
c = sp.Symbol("c", real=True)
s = sp.Symbol("s", real=True)
s1 = sp.Symbol("s_1", real=True)
s2 = sp.Symbol("s_2", real=True)
s3 = sp.Symbol("s_3", real=True)
A_abstract = A_zoh.subs({
    sp.sin(v_norm) / v_norm: s,
    sp.cos(v_norm): c,
}).subs({
    s * v.b: s1,
    s * v.c: s2,
    s * v.d: s3,
})
A_abstract

In [ ]:
tilt = sp.Quaternion(q.a, q.b, q.c, 0)
omega_tilt = sp.Quaternion(omega.a, omega.b, omega.c, omega.d)
v_tilt = sp.Quaternion(0, v.b, v.c, v.d)
display(tilt, omega_tilt, v_tilt, tilt * v_tilt)

## Numerical verification of ZoH matrix

In [ ]:
ode_update = sp.lambdify(q2l(v)[1:] + q2l(q), sp.Matrix(q2l(q * v)), modules=["numpy"])

def sci_ode_update(v, t, q):
    return np.ravel(ode_update(*np.concatenate([v, q])))

A_zoh_subs = A_zoh.subs({v_norm: sp.sqrt(v.b**2 + v.c**2 + v.d**2)})
ode_zoh = sp.lambdify(q2l(v)[1:], A_zoh_subs, modules=["numpy"])

In [ ]:
display(A_zoh_subs)

In [ ]:
@jax.jit
def ode_zoh_jax(v, dt):
    assert v.shape == (3,)
    v *= dt
    v_norm = jnp.linalg.norm(v)
    v_norm_pi = v_norm / jnp.pi
    c = jnp.cos(v_norm)
    s = jnp.sinc(v_norm_pi)
    res = jnp.array([
        [c, -v[0] * s, -v[1] * s, -v[2] * s],
        [v[0] * s, c, v[2] * s, -v[1] * s],
        [v[1] * s, -v[2] * s, c, v[0] * s],
        [v[2] * s, v[1] * s, -v[0] * s, c],
    ])
    return res

vf = np.random.uniform(-1.0, 1.0, size=3)
assert np.allclose(ode_zoh(*vf), ode_zoh_jax(vf, 1.0))

In [ ]:
np.random.seed(42)

for _ in range(2**5):
    T = np.random.uniform(1.0, 2.0)
    vf = np.random.uniform(-0.1, 0.1, size=3)
    q0f = np.random.uniform(-1.0, 1.0, size=4)
    q0f /= np.linalg.norm(q0f)
    res = sci_int.solve_ivp(
        fun=functools.partial(sci_ode_update, vf),
        t_span=[0.0, T],
        y0=q0f,
        dense_output=True,
    )

    assert np.allclose(res.sol(T),  ode_zoh(*(vf * T)) @ q0f)

## Tilt integration

Note that there is no global smooth chart on $\operatorname{Gr}(1, 4)$.

In [ ]:
v3_expr = sp.solve((tilt * v_tilt).d, v.d)[0]
v3_fun = sp.lambdify(q2l(q) + [v.b, v.c], v3_expr, modules=["numpy"])

def comp_v(q, v):
    assert q.size == 4 and v.size == 2
    v3 = v3_fun(*np.concatenate([q, v]))
    return np.concatenate([v, np.array([v3])])

v3_expr

In [ ]:
@jax.jit
def fill_v(t, v):
    assert t.shape == (3,) and v.shape == (2,)
    v3 = (-t[1] * v[1] + t[2] * v[0]) / t[0]
    return jnp.concatenate([v, jnp.array([v3])])

q0f = np.concatenate([np.random.uniform(-1.0, 1.0, size=3), np.zeros(1)])
q0f /= np.linalg.norm(q0f)
vf = np.random.uniform(-1.0, 1.0, size=2)
assert np.allclose(comp_v(q0f, vf), fill_v(q0f[:3], vf))

In [ ]:
np.random.seed(67)

for _ in range(2**10):
    T = np.random.uniform(1.0, 10.0)
    q0f = np.concatenate([np.random.uniform(-1.0, 1.0, size=3), np.zeros(1)])
    q0f /= np.linalg.norm(q0f)
    vf = np.random.uniform(-0.1, 1.0, size=2)
    vf = comp_v(q0f, vf)
    res = ode_zoh(*vf) @ q0f
    assert np.isclose(res[-1], 0.0)
    assert np.isclose(np.linalg.norm(res), 1.0)

##  Filter tilt

In [ ]:
dt = 0.01
s = ct.tf("s")
tf = 1 / (1 + s**3)
ss = tf.to_ss().sample(dt, method="zoh")
A = ss.A
B = np.ravel(ss.B)
C = np.ravel(ss.C)
D = np.squeeze(ss.D)

In [ ]:
def filt_tilt(A, B, C, D, u, x0, t0, dt):
    assert len(u.shape) == 2 and u.shape[1] == 2
    assert x0.shape == (2 * A.shape[0],)
    x0 = x0.reshape(-1, A.shape[0])

    A_bar = A + B.reshape(-1, 1) @ C.reshape(1, -1)
    B_bar = B + B * D
    x0, v0 = siso.lti_int(A_bar, B_bar, C, D, x0[0], u[:, 0])
    x1, v1 = siso.lti_int(A_bar, B_bar, C, D, x0[1], u[:, 1])
    x = jnp.hstack([x0, x1])
    v = jnp.transpose(jnp.vstack([v0, v1]))

    def comp_tilt(t, v):
        fv = fill_v(t, v)
        M = ode_zoh_jax(fv, dt)
        q = jnp.concatenate([t, jnp.zeros(1)])
        t = (M @ q)[:3]
        return t, t

    _, t = jax.lax.scan(comp_tilt, t0, v)
    return x, t

In [ ]:
np.random.seed(420)
u = np.random.uniform(0.0, 10.0, size=20).reshape(-1, 2)
x0 = np.zeros(2 * A.shape[0])
t0 = np.array([1.0, 0.0, 0.0])
assert len(filt_tilt(A, B, C, D, u, x0, t0, dt)) == 2

## Compute extra jacobians

In [ ]:
class ssincp(sp.Function):
    pass

class ssinc(sp.Function):
    def fdiff(self, argindex=1):
        return ssincp(self.args[0])

class scos(sp.Function):
    def fdiff(self, argindex=1):
        frac = -1 / sp.sympify(2)
        return frac * ssinc(self.args[0])

In [ ]:
t = sp.Matrix([[sp.Symbol("t_0")], [sp.Symbol("t_1")], [sp.Symbol("t_2")]])
ta = sp.Array([t[0, 0], t[1, 0], t[2, 0]])
dt_sp = sp.Symbol("dt")
sub_q2t = {
    q.a: t[0, 0],
    q.b: t[1, 0],
    q.c: t[2, 0],
}

M = A_zoh_subs.subs({
    sp.cos(sp.sqrt(v_norm_2)): scos(v_norm_2),
    sp.sin(sp.sqrt(v_norm_2)) / sp.sqrt(v_norm_2): ssinc(v_norm_2),
}).subs({
    v.b: v.b * dt_sp,
    v.c: v.c * dt_sp,
    v.d: v.d * dt_sp,
})
M_subs = M.subs({v.d: v3_expr})[:3, :3]
g = M_subs * t
g

In [ ]:
v3 = (-t[1, 0] * v.c + t[2, 0] * v.b)
ssinc_in = dt_sp**2 * v.b**2 + dt_sp**2 * v.c**2 + dt_sp**2 * v3**2 / t[0, 0]**2
sinc_sp = [sp.Symbol("c_0"), sp.Symbol("s_0"), sp.Symbol("s_1")]
dt_pow = [sp.Symbol("dt_1"), sp.Symbol("dt_2"), sp.Symbol("dt_3")]
sinc_subs = {
    scos(ssinc_in): sinc_sp[0],
    ssinc(ssinc_in): sinc_sp[1],
    ssincp(ssinc_in): sinc_sp[2],
    v3: v.d * t[0, 0],
}
dt_subs = {
    dt_sp: dt_pow[0],
    dt_sp**2: dt_pow[1],
    dt_sp**3: dt_pow[2],
}

In [ ]:
g0a = sp.Array([[[
    sp.cancel(sp.expand(M_subs[i, k].diff(q2l(q)[j]).subs(sub_q2t).subs(sinc_subs))).subs(dt_subs)
    for k in range(3)] for j in range(3)] for i in range(3)]
)
check_g0a = sp.tensorcontraction(sp.tensorproduct(g0a, ta), (2, 3))
check_g0a = sp.cancel(sp.expand(sp.Matrix(check_g0a)))
g0a

In [ ]:
g0 = sp.Matrix(3, 3, lambda i, j: g[i].diff(q2l(q)[j])).subs(sub_q2t)
g0 = sp.cancel(sp.expand(g0.subs(sinc_subs))).subs(dt_subs)
g0

In [ ]:
check_g0a - g0

In [ ]:
g1a = sp.Array([[[
    sp.cancel(sp.expand(M_subs[i, k].diff(q2l(v)[1:3][j]).subs(sub_q2t).subs(sinc_subs))).subs(dt_subs)
    for k in range(3)] for j in range(2)] for i in range(3)]
)
check_g1a = sp.tensorcontraction(sp.tensorproduct(g1a, ta), (2, 3))
check_g1a = sp.cancel(sp.expand(sp.Matrix(check_g1a)))
g1a

In [ ]:
g1 = sp.Matrix(3, 2, lambda i, j: g[i].diff(q2l(v)[1:][j])).subs(sub_q2t)
g1 = sp.cancel(sp.expand(g1.subs(sinc_subs))).subs(dt_subs)
g1

In [ ]:
check_g1a - g1

In [ ]:
g2 = sp.Matrix(3, 3, lambda i, j: g[i].diff(t[j, 0])).subs(sub_q2t)
g2 = sp.cancel(sp.expand(g2.subs(sinc_subs))).subs(dt_subs)
g2

In [ ]:
# g_lam = sp.lambdify(list(t) + q2l(v)[1:4] + [dt_pow] + sinc_sp, (g0, g1, g2), modules=["jax"], cse=True, docstring_limit=None)
# g_lam = sp.lambdify(list(t) + q2l(v)[1:4] + dt_pow + sinc_sp, (g0, g1, g2), modules=["jax"], cse=True, docstring_limit=None)
g_lam = sp.lambdify(list(t) + q2l(v)[1:4] + dt_pow + sinc_sp, (sp.simplify(g0a * t[0, 0]), sp.simplify(g1a* t[0, 0]), g2), modules=["jax"], cse=True, docstring_limit=None)
print(g_lam.__doc__)

In [ ]:
# remark: the tensor implementation is not faster, it is only more obvious
#  to me that it is properly reusing as much computation as possible
# It is the same speed as the (simpler) monolithic implementation

def ssinc_deriv(x):
    sqr = jnp.sqrt(x)
    scos = jnp.cos(sqr)
    ssinc = jnp.sin(sqr) / sqr
    ssincp = (scos - ssinc) / (2 * x)
    return jax.lax.cond(
        x <= 0,
        lambda: (1.0, 1.0, -1 / 6),
        lambda: (scos, ssinc, ssincp),
    )

def g_tensor(t_0, t_1, t_2, v_1, v_2, v_3, dt_1, dt_2, dt_3, c_0, s_0, s_1):
    x0 = v_3**2
    x1 = dt_2*s_0
    x2 = x0*x1
    x3 = 2*dt_3*s_1
    x4 = x0*x3
    x5 = v_1*x4
    x6 = v_2*x4
    x7 = v_3*x1
    x8 = v_2*x7
    x9 = v_3*x3
    x10 = v_1*v_2*x9
    x11 = v_2**2
    x12 = x11*x9
    x13 = -v_1*x7
    x14 = v_1**2
    x15 = x14*x9
    x16 = -x10
    x17 = dt_1*s_0
    x18 = x17 + x4
    x19 = -x18
    x20 = t_0*v_1
    x21 = t_2*v_3
    x22 = x20 + x21
    x23 = -x22
    x24 = x1*x23
    x25 = t_0*x17
    x26 = v_1*x3
    x27 = t_0*x3
    x28 = x14*x27 + x21*x26 + x25
    x29 = v_2*x3
    x30 = t_0*v_2 - t_1*v_3
    x31 = -x30
    x32 = x1*x31
    x33 = -2*dt_3*s_1*t_1*v_2*v_3 + x11*x27 + x25
    x34 = t_2*x17 + t_2*x4 + x20*x9
    x35 = -2*dt_3*s_1*t_0*v_2*v_3 + t_1*x17 + t_1*x4
    x36 = v_1*x17
    x37 = v_2*x17
    x38 = v_3*x17
    g0a = jnp.array([[[x2, x5, x6], [x8, x10, x12], [x13, -x15, x16]], [[-x5, x2, v_3*x19], [x16, x8, v_2*x19], [x15, x13, v_1*x18]], [[-x6, v_3*x18, x2], [-x12, v_2*x18, x8], [x10, v_1*x19, x13]]])
    g1a = jnp.array([[[x24, -x28, x23*x29], [x32, x26*x31, -x33]], [[x28, x24, x34], [x26*x30, x32, -x35]], [[x22*x29, -x34, x24], [x33, x35, x32]]])
    g2 = jnp.array([[c_0, -x36, -x37], [x36, c_0, x38], [x37, -x38, c_0]])
    return g0a, g1a, g2

@jax.jit
def g_deriv(t, v, dt):
    assert t.shape == (3,) and v.shape == (2,)
    dt_1 = dt
    dt_2 = dt_1 * dt
    dt_3 = dt_2 * dt
    v3 = (-t[1] * v[1] + t[2] * v[0]) / t[0]
    v_square = v[0]**2 + v[1]**2 + v3**2
    c_0, s_0, s_1 = ssinc_deriv(v_square * dt_2)
    g0a, g1a, g2 = g_tensor(t[0], t[1], t[2], v[0], v[1], v3, dt_1, dt_2, dt_3, c_0, s_0, s_1)
    
    g0 = jnp.tensordot(g0a / t[0], t, axes=([2], [0]))
    g1 = jnp.tensordot(g1a / t[0], t, axes=([2], [0]))
    return g0, g1, g2

In [ ]:
import exp_mpc.stewart_min.comp as comp
import exp_mpc.stewart_min.mpc_spec as mpc_spec

spec = mpc_spec.MPCSpec()

def g_jax(x1, x2, x3):
    fv = comp.fill_v(x1, x2)
    M = comp.tilt_zoh(fv, spec.dt)
    return M @ x3

@jax.jit
def g_deriv_jax(t, v):
    return jax.jacrev(g_jax, argnums=[0, 1, 2])(t, v, t)

In [ ]:
np.random.seed(42)
qf = np.random.uniform(-1.0, 1.0, size=3)
qf /= np.linalg.norm(qf)
vf = np.random.uniform(-1.0, 1.0, size=2)
# vf = np.zeros(2)

check0 = g_deriv(qf, vf, spec.dt)
check1 = g_deriv_jax(qf, vf)
max(jnp.abs(jnp.max(check0[idx]- check1[idx])) for idx in range(len(check0)))

## old

In [ ]:
def old_g_deriv(t, v, dt):
    assert t.shape == (3,) and v.shape == (2,)
    t_0, t_1, t_2 = t
    v_1, v_2 = v

    x0 = t_0**(-3)
    x1 = t_1*v_2 - t_2*v_1
    x2 = x1**2
    x3 = t_0**2
    x4 = dt**2
    x5 = x3**(-1)
    x6 = x4*x5
    x7 = x6*(x2 + x3*(v_1**2 + v_2**2))

    c0, s0, s1 = ssinc_deriv(x7)

    x8 = s0
    x9 = t_0*x8
    x10 = t_1*v_1 + t_2*v_2
    x11 = s1
    x12 = 2*x11
    x13 = dt*x12
    x14 = x10*x13 + x9
    x15 = x2*x4
    x16 = -x10
    x17 = x1*x6
    x18 = x12*x4
    x19 = x1*x18
    x20 = -t_2*x8 + v_1*x19
    x21 = x12*x15
    x22 = t_1*x1
    x23 = dt*x9
    x24 = t_2*x21 + x22*x23
    x25 = -x20*x3 + x24
    x26 = t_0**4
    x27 = dt*x1/x26
    x28 = dt*x0
    x29 = v_2*x28
    x30 = v_1*x28
    x31 = t_1*x8
    x32 = -dt*t_0*t_2*x1*x8 + t_1*x21 + x3*(v_2*x19 + x31)
    x33 = -x32
    x34 = t_2*x1
    x35 = v_1*x3 - x34
    x36 = x23*x35
    x37 = dt*x5
    x38 = t_2*x8
    x39 = v_2*x3 + x22
    x40 = x23*x39
    x41 = x18*x34
    x42 = v_1*x18
    x43 = -t_1*x38
    x44 = x18*x22
    x45 = v_2*x18
    x46 = c0
    x47 = dt*x8
    x48 = v_1*x47
    x49 = v_2*x47
    x50 = x1*x47/t_0
    g0 = jnp.array([[x0*x14*x15, v_2*x17*(x13*x16 - x9), v_1*x14*x17], [x25*x27, x29*(x20*x3 - x24), x25*x30], [x27*x33, x29*x32, x30*x33]])
    g1 = jnp.array([[x37*(2*x11*x16*x35*x4 - x3*x31 - x36), x37*(2*x11*x16*x39*x4 - x3*x38 - x40)], [x28*(-t_1*x36 + x26*x8 + x3*(t_2**2*x8 + x35*x42) - x35*x41), x28*(-t_1*x40 + x3*(x39*x42 + x43) - x39*x41)], [x28*(-t_2*x36 + x3*(x35*x45 + x43) + x35*x44), x28*(-t_2*x40 + x26*x8 + x3*(t_1**2*x8 + x39*x45) + x39*x44)]])
    g2 = jnp.array([[x46, -x48, -x49], [x48, x46, -x50], [x49, x50, x46]])
    return (g0, g1, g2)
